|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>CUDA graphs<h1>|
|<h2>Lecture:</h2>|<h1><b>A graph refuses to change shape, and a server does nothing else<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# What a graph refuses to do

Capture froze everything: the pointers, the shapes, the control flow. Replay
runs exactly what was recorded.

A server does not have that luxury. The batch size changes every step, because
that was the entire point of continuous batching.

In [ ]:
MODEL = """class Block(nn.Module):
  \"\"\"One transformer layer, shaped like a decode step: many small kernels.\"\"\"
  def __init__(self, d):
    super().__init__()
    self.n1  = nn.RMSNorm(d)
    self.qkv = nn.Linear(d, 3*d, bias=False)
    self.o   = nn.Linear(d, d, bias=False)
    self.n2  = nn.RMSNorm(d)
    self.up  = nn.Linear(d, 4*d, bias=False)
    self.dn  = nn.Linear(4*d, d, bias=False)

  def forward(self, x):
    h = self.n1(x)
    q, k, v = self.qkv(h).chunk(3, -1)
    a = torch.softmax(q @ k.transpose(-1,-2) / 32.0, -1) @ v
    x = x + self.o(a)
    h = self.n2(x)
    x = x + self.dn(F.silu(self.up(h)))
    return x

class Model(nn.Module):
  def __init__(self, n_layers=28, d=1024):
    super().__init__()
    self.blocks = nn.ModuleList([Block(d) for _ in range(n_layers)])
  def forward(self, x):
    for b in self.blocks: x = b(x)
    return x"""

In [ ]:
CAPTURE = """def capture(model, example):
  \"\"\"Record one forward pass as a graph, and return (graph, input, output).

  The warm-up on a side stream is not optional: cuBLAS and friends allocate
  workspaces on first use, and you must not record that.
  \"\"\"
  static_in = example.clone()
  s = torch.cuda.Stream()
  s.wait_stream(torch.cuda.current_stream())
  with torch.cuda.stream(s):
    for _ in range(3):
      model(static_in)
  torch.cuda.current_stream().wait_stream(s)

  g = torch.cuda.CUDAGraph()
  with torch.cuda.graph(g):
    static_out = model(static_in)
  return g, static_in, static_out"""

In [ ]:
exec(MODEL); exec(CAPTURE)

D, LAYERS = 1024, 28
model = Model(LAYERS, D).cuda().to(torch.bfloat16).eval()

with torch.no_grad():
  x4 = torch.randn(4, 1, D, device='cuda', dtype=torch.bfloat16)
  g, static_in, static_out = capture(model, x4)
print(f'captured at batch {static_in.shape[0]}')

In [ ]:
# now a step arrives with 3 sequences instead of 4
x3 = torch.randn(3, 1, D, device='cuda', dtype=torch.bfloat16)
try:
  static_in.copy_(x3)
except RuntimeError as e:
  print('copy_ refused:', str(e).splitlines()[0])

### The fix is the same one JAX needs, for a different reason

Capture a graph for each of a handful of **bucketed** batch sizes, and pad the
real batch up to the next bucket. A step with 3 sequences runs the graph for
4, with one row of padding that produces a token nobody reads.

In [ ]:
BUCKETS = [1, 2, 4, 8, 16, 32]

graphs = {}
with torch.no_grad():
  for b in BUCKETS:
    xb = torch.randn(b, 1, D, device='cuda', dtype=torch.bfloat16)
    graphs[b] = capture(model, xb)
print(f'captured {len(graphs)} graphs: {BUCKETS}')

def bucket_for(n):
  return next((b for b in BUCKETS if b >= n), None)

for n in (1, 3, 5, 12, 31, 40):
  b = bucket_for(n)
  waste = (b-n)/b if b else None
  print(f'{n:>3} sequences -> bucket {str(b):>4}' +
        (f', {100*waste:4.0f}% of the rows are padding' if b else ', no graph, run eager'))

### And now measure whether it was worth it

In [ ]:
def run_bucketed(n):
  b = bucket_for(n)
  g, si, so = graphs[b]
  si[:n].copy_(torch.randn(n, 1, D, device='cuda', dtype=torch.bfloat16))
  g.replay()
  return so[:n]

print(f"{'seqs':>5} {'eager ms':>10} {'bucketed ms':>12} {'speedup':>8} {'padding':>8}")
for n in (1, 3, 5, 12, 31):
  xn = torch.randn(n, 1, D, device='cuda', dtype=torch.bfloat16)
  with torch.no_grad():
    e = cudalib.bench_ms(lambda: model(xn), iters=50, warmup=20, best_of=2)
  gms = cudalib.bench_ms(lambda: run_bucketed(n), iters=50, warmup=20, best_of=2)
  b = bucket_for(n)
  print(f'{n:>5} {e:>10.3f} {gms:>12.3f} {e/gms:>7.2f}x {100*(b-n)/b:>7.0f}%')

### Two costs, and only one of them is obvious

**Padding.** A step with 5 sequences runs the graph for 8, so three rows of
arithmetic are thrown away. On the memory-bound left half of the roofline
those rows are nearly free, which is the only reason this works at all.

**Memory.** Every captured graph holds its own input, output and intermediate
buffers for the lifetime of the server. Six buckets is six copies. That is
memory the KV pool does not get, so the bucket list is a real trade against
the thing Part 3 spent four sections protecting.

### The same fix, arrived at twice

On the JAX track there are no kernel launches to remove: XLA already fused
the step into one graph. What JAX has instead is **recompilation**, seconds of
it, every time a new batch size appears. The fix is identical: pad to bucketed
shapes so the compilation cache hits.

Same answer, completely different reason, which is the strongest evidence
that bucketing is not a CUDA trick. It is what you do whenever the cost of
preparing to run work is large compared with the work.

    ./vc guide 12